# Maritaca Hybrid Graph Agentic

Built with Brazilian LLMs, for Brazilian AI engineering.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Literal

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from openai import OpenAI
from pydantic import BaseModel

import json
import os
import time

In [ ]:
load_dotenv(dotenv_path=Path(".env"))

MODEL = "sabiazinho-4"

client = OpenAI(
    base_url="https://chat.maritaca.ai/api",
    api_key=os.getenv("MARITACA_API_KEY"),
)

In [ ]:
class ToolScore(BaseModel):
    tool: Literal["cag", "rag"]
    expected_value: float
    estimated_cost: float
    reason: str


class GraphPlan(BaseModel):
    selected_tools: list[Literal["cag", "rag"]]
    scores: list[ToolScore]
    reason: str


class JudgeDecision(BaseModel):
    is_enough: bool
    reason: str
    missing_info: str | None = None

In [ ]:
graph_state = {
    "question": None,
    "plan": None,
    "tool_results": [],
    "aggregated_context": None,
    "judge": None,
    "final_answer": None,
    "logs": [],
}

In [ ]:
def extract_json(raw_text: str) -> dict:
    json_start = raw_text.find("{")
    json_end = raw_text.rfind("}")

    if json_start == -1 or json_end == -1:
        raise ValueError(f"Resposta sem JSON valido: {raw_text}")

    return json.loads(raw_text[json_start:json_end + 1])

In [ ]:
def plan_node(question: str) -> GraphPlan:
    start = time.time()

    prompt = f"""
PAPEL:
Voce e o PLANNER de um sistema chamado Maritaca Hybrid Graph Agentic.

OBJETIVO:
Escolher quais ferramentas devem ser usadas para responder a pergunta do usuario.

CONTEXTO:
O sistema possui duas memorias/ferramentas:
- CAG para conhecimento cacheado.
- RAG para documentos e evidencias textuais.

OPCOES DISPONIVEIS:
- cag
- rag

REGRAS:
- Se a pergunta for simples e ligada a FAQ, regras, horarios, planos ou precos, escolha [\"cag\"].
- Se a pergunta pedir documentos, contratos, PDFs, manuais ou evidencias textuais, escolha [\"rag\"].
- Se a pergunta precisar comparar conhecimento cacheado com documentos, escolha [\"cag\", \"rag\"].
- Nao escolha todas as ferramentas sem necessidade.
- Sempre gere score para cag e rag.
- expected_value deve ir de 0.0 a 1.0.
- estimated_cost deve ir de 0.0 a 1.0.

ENTRADA:
Pergunta do usuario:
{question}

FORMATO DE SAIDA:
Responda APENAS JSON valido.
Nao use markdown.
Nao escreva texto antes ou depois do JSON.
O JSON deve ter exatamente este formato:
{{
  \"selected_tools\": [\"cag\"],
  \"scores\": [
    {{\"tool\": \"cag\", \"expected_value\": 0.9, \"estimated_cost\": 0.1, \"reason\": \"motivo curto\"}},
    {{\"tool\": \"rag\", \"expected_value\": 0.2, \"estimated_cost\": 0.4, \"reason\": \"motivo curto\"}}
  ],
  \"reason\": \"motivo geral da selecao\"
}}
"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
    )

    raw_text = response.choices[0].message.content
    data = extract_json(raw_text)
    plan = GraphPlan(**data)

    graph_state["logs"].append({
        "node": "plan_node",
        "elapsed": round(time.time() - start, 2),
        "input": {"question": question},
        "output": plan.model_dump(),
    })

    return plan

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db = Chroma(
    persist_directory="chroma_db",
    embedding_function=embeddings,
)

In [ ]:
def cag_node(question: str) -> dict:
    start = time.time()

    cache = json.loads(Path("cache.json").read_text(encoding="utf-8"))

    prompt = f"""
PAPEL:
Voce e a ferramenta CAG.

OBJETIVO:
Buscar no cache de conhecimento informacoes relevantes para responder a pergunta.

CONTEXTO:
{json.dumps(cache, ensure_ascii=False, indent=2)}

REGRAS:
- Use apenas o cache.
- Se nao encontrar informacao relevante, diga isso claramente.
- Responda em portugues do Brasil.
- Seja objetivo.

ENTRADA:
Pergunta do usuario:
{question}

FORMATO DE SAIDA:
Texto normal com as informacoes relevantes encontradas no cache.
"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
    )

    output = {
        "tool": "cag",
        "query": question,
        "result": response.choices[0].message.content,
        "elapsed": round(time.time() - start, 2),
    }

    graph_state["logs"].append({
        "node": "cag_node",
        "elapsed": output["elapsed"],
        "output": output,
    })

    return output


def rag_node(question: str) -> dict:
    start = time.time()

    docs = db.similarity_search(question, k=3)
    rag_context = "\n\n".join([doc.page_content for doc in docs])

    output = {
        "tool": "rag",
        "query": question,
        "result": rag_context,
        "elapsed": round(time.time() - start, 2),
    }

    graph_state["logs"].append({
        "node": "rag_node",
        "elapsed": output["elapsed"],
        "output": output,
    })

    return output

In [ ]:
TOOL_NODES = {
    "cag": cag_node,
    "rag": rag_node,
}

In [ ]:
def run_parallel_tools(question: str, selected_tools: list[str]) -> list[dict]:
    start = time.time()
    results = []

    with ThreadPoolExecutor(max_workers=max(1, len(selected_tools))) as executor:
        futures = {
            executor.submit(TOOL_NODES[tool], question): tool
            for tool in selected_tools
        }

        for future in as_completed(futures):
            tool = futures[future]

            try:
                result = future.result()
            except Exception as error:
                result = {
                    "tool": tool,
                    "query": question,
                    "result": f"Erro ao executar {tool}: {error}",
                    "elapsed": None,
                    "error": True,
                }

            results.append(result)

    graph_state["logs"].append({
        "node": "parallel_tools",
        "elapsed": round(time.time() - start, 2),
        "input": {
            "selected_tools": selected_tools,
            "question": question,
        },
        "output": results,
    })

    return results

In [ ]:
def aggregator_node(tool_results: list[dict]) -> str:
    start = time.time()
    parts = []

    for item in tool_results:
        parts.append(
            f"""
FONTE: {item["tool"]}
QUERY: {item["query"]}
RESULTADO:
{item["result"]}
"""
        )

    aggregated_context = "\n---\n".join(parts)

    graph_state["logs"].append({
        "node": "aggregator_node",
        "elapsed": round(time.time() - start, 2),
        "input": tool_results,
        "output": aggregated_context,
    })

    return aggregated_context

In [ ]:
def judge_node(question: str, aggregated_context: str) -> JudgeDecision:
    start = time.time()

    prompt = f"""
PAPEL:
Voce e um juiz de suficiencia.

TAREFA:
Verificar se o contexto abaixo e suficiente para responder a pergunta do usuario.

IMPORTANTE:
Voce NAO deve responder a pergunta.
Voce deve apenas avaliar se da para responder com o contexto disponivel.

PERGUNTA:
{question}

CONTEXTO:
{aggregated_context}

REGRAS:
- Se o contexto contem a resposta, use is_enough true.
- Se falta informacao importante, use is_enough false.
- Se a pergunta pede comparacao com documentos e a fonte RAG nao traz evidencia relevante, use is_enough false.
- Se uma fonte trouxer textos irrelevantes ao tema da pergunta, nao trate isso como evidencia suficiente.
- Nao invente informacoes.
- Se is_enough for true, missing_info deve ser null.
- Se is_enough for false, missing_info deve dizer o que falta.

RESPONDA APENAS JSON VALIDO NESTE FORMATO:
{{
  \"is_enough\": true,
  \"reason\": \"motivo curto\",
  \"missing_info\": null
}}
"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
    )

    raw_text = response.choices[0].message.content
    data = extract_json(raw_text)
    judge = JudgeDecision(**data)

    graph_state["logs"].append({
        "node": "judge_node",
        "elapsed": round(time.time() - start, 2),
        "input": {
            "question": question,
            "aggregated_context": aggregated_context,
        },
        "output": judge.model_dump(),
    })

    return judge

In [ ]:
def final_node(question: str, aggregated_context: str) -> str:
    start = time.time()

    prompt = f"""
PAPEL:
Voce e o Decisor Final.

OBJETIVO:
Responder a pergunta do usuario usando apenas o contexto fornecido.

CONTEXTO:
{aggregated_context}

REGRAS:
- Responda em portugues do Brasil.
- Use apenas informacoes presentes no contexto.
- Nao invente informacoes.
- Se o contexto nao tiver informacao suficiente, diga isso claramente.
- Seja claro e direto.

ENTRADA:
Pergunta do Usuario:
{question}

FORMATO DE SAIDA:
Responda em texto normal, sem JSON ou markdown.
"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
    )

    raw_text = response.choices[0].message.content

    graph_state["logs"].append({
        "node": "final_node",
        "elapsed": round(time.time() - start, 2),
        "input": {
            "question": question,
            "aggregated_context": aggregated_context,
        },
        "output": raw_text,
    })

    return raw_text

In [ ]:
def run_graph(question: str) -> str:
    graph_state["question"] = question
    graph_state["plan"] = None
    graph_state["tool_results"] = []
    graph_state["aggregated_context"] = None
    graph_state["judge"] = None
    graph_state["final_answer"] = None
    graph_state["logs"] = []

    plan = plan_node(question)
    graph_state["plan"] = plan.model_dump()

    tool_results = run_parallel_tools(
        question=question,
        selected_tools=plan.selected_tools,
    )
    graph_state["tool_results"] = tool_results

    aggregated_context = aggregator_node(tool_results)
    graph_state["aggregated_context"] = aggregated_context

    judge = judge_node(question, aggregated_context)
    graph_state["judge"] = judge.model_dump()

    if not judge.is_enough:
        answer = f"Contexto insuficiente: {judge.missing_info}"
        graph_state["final_answer"] = answer
        return answer

    answer = final_node(question, aggregated_context)
    graph_state["final_answer"] = answer

    return answer

In [ ]:
def print_graph_logs() -> None:
    for log in graph_state["logs"]:
        print(log["node"], log["elapsed"])

In [ ]:
answer = run_graph("Qual o horario de suporte?")
print(answer)
print_graph_logs()